[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/magnetic_resonance/magnetic_resonance_spacetime.ipynb)

# Magnetic Resonance in Spacetime

The experiment of [the magnetic resonance notebook](magnetic_resonance.ipynb), a driven, relaxing spin, its absorption line and its echo, computed in the algebra of spacetime instead of the algebra of space. A spin's state is defined relative to an observer: its Bloch vector is a plane through the observer's time direction `t`, such as `mv.zt`, which the observer sees as the direction z, and every object here, state, field, relaxation and pulse, lies in the even algebra. That observer is the lab: the common rest frame of the spin, the magnet and the bath, placed by a rotor. It enters wherever a direction becomes a plane through its time direction, and in the adjoint `observer >> process.reverse()`. Bound to it, the relaxation, the generator, the evolution and the pulses are maps from state to state, as they are in space.

Times are in microseconds and frequencies in radians per microsecond, with `T1` = 10 µs and `T2` = 4 µs, typical of an electron spin in a solid.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from collections.abc import Generator, Iterator

import numpy as np
from IPython.display import Image, display

from numga import NumpyContext, stack
from numga.algebras import STA
from examples.animation import save_animation
from examples.quantum.magnetic_resonance import render

np.set_printoptions(precision=3, suppress=True)

# The algebra of spacetime, with signature (+, -, -, -).
ga = STA
mv = NumpyContext(ga).multivector
# Observers are vectors. The Bloch vector and the field are planes through the observer's time
# direction, which the observer sees as directions in space; relaxation processes are planes.
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
# A spin's state is even: one half plus the Bloch vector, a plane through the observer's time, and
# relaxation can reach every even part: the scalar, the six planes and the pseudoscalar.
State = ga.gatype.even()                               # 1 tx ty xy tz xz yz txyz
Rates = ga.gatype((State, State))                      # State <- State
# What becomes of each state over a stretch of time.
Evolution = ga.gatype((State, State))                  # State <- State
# A relaxation process is a plane: one through the observer's time plus I times another.
Process = ga.gatype.bivector()
Rotor = ga.gatype.rotor()
# The pseudoscalar squares to minus one and commutes with every even element.
I = mv.txyz                                            # [] Pseudoscalar
one = mv.scalar([1.0])                                 # [] Scalar
# The lab at rest: the rest frame of the spin, the magnet and the bath.
rest = mv.rotor()                                      # [] Rotor
T1, T2 = 10.0, 4.0                                     # µs

## 1. The state and how it changes

The state is `state(bloch)`, half of one plus the Bloch vector, now a plane through the observer's time direction such as `mv.zt`. The field is such a plane too, and its dual, the plane it turns the spin in, turns the state by commutation. A relaxation `process` changes the state `rho` by `process * rho * adjoint` less half of `adjoint * process` multiplied onto the state from either side, with the adjoint `observer >> process.reverse()`. Raising toward the field is the plane `xt` times the state against the field: the plane `(xt + I * yt) / 2`, which squares to zero. Leaving the state open and binding the observer gives the generator, a map on the even algebra.

In matrix notation the state reads as a 2×2 Hermitian density matrix, the adjoint as the Hermitian conjugate, the planes `xt`, `yt` and `zt` as the Pauli matrices, $I$ as the imaginary unit $i$, and the generator as the Lindblad superoperator, $\dot\rho = -i[H, \rho] + \sum_k \big(L_k \rho L_k^\dagger - \tfrac12 \{L_k^\dagger L_k, \rho\}\big)$.

In [ ]:
def state(bloch: Bivector) -> State:
    """The state with the given Bloch vector, a plane through the observer's time direction."""
    return 0.5 * (one + bloch)                                                  # [...] State


def relaxation(process: Process, observer: Vector) -> Rates:
    """The change of a state due to a relaxation process, as a map: the state is left open. The
    observer's sandwich of the reverse is the adjoint."""
    adjoint = observer >> process.reverse()                                     # [] Bivector
    back = adjoint * process                                                    # [] State
    return process * State * adjoint - back.anticommutator(State)               # State <- State


def generator(detuning: np.ndarray, drive: np.ndarray, lab: Rotor) -> Rates:
    """The rate of change of a state, in the frame that turns with the drive, for a lab placed by a
    rotor: its time direction is the observer, the field lies along its z and the drive along its x."""
    observer, x, z = lab >> mv.t, lab >> mv.x, lab >> mv.z                      # [] Vector each
    field = (z * detuning + x * drive) ^ observer                               # [...] Bivector
    # The plane the field turns the spin in is its dual.
    turning = field.dual().commutator(State)                                    # [...] State <- State
    # The plane x times the state of a spin against the field: it turns that part of the state over
    # onto the field.
    raise_toward_field = (x ^ observer) * state(-(z ^ observer))                # [] Process
    relaxing = (1 / T1) * relaxation(raise_toward_field, observer) + 0.5 * (1 / T2 - 1 / (2 * T1)) * relaxation(z ^ observer, observer)   # State <- State
    return turning + relaxing                                                   # [...] State <- State


# A detuning of 0.7 and a drive of 1.3 rad/µs, acting on a state.
rates = generator(0.7, 1.3, rest)                                               # [] State <- State
# A direction as the lab at rest sees it: wedged with its time, any part along that time drops out.
bloch = mv.vector([0.0, 0.4, -0.2, 0.6]) ^ mv.t                                 # [] Bivector
change = rates(state(bloch))                                                    # [] State

In [ ]:
# The change of the Bloch vector:
print("generator:", render.components(change))

## 2. The steady state

Under a steady drive the spin spirals into a state the generator sends to zero. The generator keeps the centre of the even algebra, the scalar and the pseudoscalar, so both are pinned: adding them as dyads on one and on I makes the map invertible, and a single solve against one half gives the steady state. The evolution over a short time `dt` is the exponential series of the generator, as in space.

In matrix notation the pinned centre reads as the trace of the density matrix, real and imaginary part; the evolution over $dt$ as $e^{\mathcal{L}\,dt}$ to fourth order.

In [ ]:
def steady(rates: Rates) -> State:
    """The state that no longer changes, with its scalar part pinned to one half and its
    pseudoscalar part to zero."""
    centre = one * one.scalar_product(State) + I * I.scalar_product(State)     # State <- State
    return (rates + centre).solve(one * 0.5)                                    # [...] State


def evolution(rates: Rates, dt: float) -> Evolution:
    """What becomes of each state over a time dt: exp(dt L), to fourth order in dt."""
    small = rates * dt                                                          # [...] State <- State
    term = total = State
    for order in range(1, 5):
        # The next term of the exponential series: the last one composed with small, over the order.
        term = small(term) / order                                              # [...] State <- State
        total = total + term
    return total                                                                # [...] State <- State


def evolve(each_step: Evolution, rho: State, steps: int) -> Generator[State, None, State]:
    """The state before each of the given number of steps; returns the state after the last."""
    for _ in range(steps):
        yield rho                                                               # [...] State
        rho = each_step(rho)
    return rho


detunings = np.array([0.0, 0.5, 1.0])                                           # rad/µs
driven = generator(detunings, 1.0, rest)                                        # [detunings] State <- State
# All spins start along the field and are driven for 40 µs.
along_field = state(mv.zt).broadcast_to(detunings.shape)                        # [detunings] State
settled = steady(driven)                                                        # [detunings] State
render.draw_nutation(evolve(evolution(driven, 0.02), along_field, 2000), settled);

## 3. Line shapes

The absorption is the part of the steady Bloch vector a quarter turn behind the drive, along minus `yt`, the observer's direction y, and the dispersion the part in step with it, along `xt`. The whole spectrum for several drive strengths is one batched solve; driven harder, the line saturates and broadens.

In [ ]:
sweep = np.linspace(-3.0, 3.0, 241)                                              # detunings, rad/µs
drives = np.array([0.05, 0.3, 1.0])                                              # rad/µs
spectrum = steady(generator(sweep[:, None], drives[None, :], rest))              # [sweep, drives] State
render.draw_lines(sweep, drives, spectrum);

## 4. Spin echo

A short pulse is a rotor in a spacelike plane, `I * mv.xt`. Such a plane commutes with the observer, so the observer's sandwich of the rotor's reverse is the reverse itself, and a pulse acts on a state by its sandwich, as in space. The spins, each with its own detuning, fan out after the first pulse and line up again after the second, having lost only the true dephasing.

In the notation of NMR the Hahn echo amplitude at twice the delay $\tau$ reads as $e^{-2\tau/T_2}$.

In [ ]:
def pulse(angle: float) -> Rotor:
    """A short strong pulse along the plane xt, the observer's direction x, turning the Bloch vector
    about it by the angle."""
    return (I * mv.xt * (-angle / 2)).exp()                                     # [] Rotor


def echo(each_step: Evolution, rho: State, before: int, after: int) -> Iterator[State]:
    """The states through a spin echo: tipped onto -y, left for the given number of steps, turned
    half a turn about x, and left again."""
    rho = yield from evolve(each_step, pulse(np.pi / 2) >> rho, before)
    yield from evolve(each_step, pulse(np.pi) >> rho, after)


spread, delay, dt = 2.0, 2.5, 0.01                                               # rad/µs, µs, µs
scattered = np.random.default_rng(0).normal(0.0, spread, 300)                    # [spins] rad/µs
# No drive between the pulses; the spins start along the field and are followed for 8 µs in all.
free = generator(scattered, 0.0, rest)                                           # [spins] State <- State
along_field = state(mv.zt).broadcast_to(scattered.shape)                         # [spins] State
waits = round(delay / dt)
# The spins seen from above the field, one frame every 8 steps, beside the signal so far.
frames = render.animate_echo(scattered, echo(evolution(free, dt), along_field, waits, 800 - waits), dt, delay, T2, 8)
display(Image(filename=save_animation(frames, "resonance_echo_spacetime", 40)))

## 5. The experiment as one map

The evolution over one step, doubled by composition, and the pulses, their sandwiches with the state left open, compose into one map from state to state for the whole echo experiment. Averaged over the spins it is the sample's map, built for delays from 10 ns to 10 µs at once.

In [ ]:
def doublings(span: Evolution, count: int) -> Iterator[Evolution]:
    """The evolution over its own span, then over twice and four times that span, and so on."""
    for _ in range(count):
        yield span
        span = span(span)


# The evolution over one step, doubled for delays from dt to 1024 dt.
waiting = stack(list(doublings(evolution(free, dt), 11)))                       # [delays, spins] State <- State
# The times after the first pulse.
times = 2 * dt * 2.0 ** np.arange(11)                                            # µs
# The pulses as maps: their sandwiches with the state left open.
tip = pulse(np.pi / 2) >> State                                                  # [] State <- State
turn = pulse(np.pi) >> State                                                     # [] State <- State
# Tip, wait, turn, wait: the echo. Tip and wait twice as long: the free decay.
echo = waiting(turn(waiting(tip)))                                               # [delays, spins] State <- State
decay = waiting(waiting(tip))                                                    # [delays, spins] State <- State
# Every spin has its own map; the sample's is their average.
sample_echo = echo.mean(axis=-1)                                                 # [delays] State <- State
sample_decay = decay.mean(axis=-1)                                               # [delays] State <- State
echoed = sample_echo(state(mv.zt))                                               # [delays] State
faded = sample_decay(state(mv.zt))                                               # [delays] State
render.draw_decays(times, echoed, faded, T2);

## 6. The lab is the observer

Carry the whole lab along, spin, magnet, drive and bath together, by a boost: every plane of the setup is built from the lab's own directions, so the steady state is carried along with it and the physics is unchanged. A moving observer reading the lab at rest is another matter: the planes through the lab's time are not all planes through the moving observer's time, so part of the state is not Hermitian for that observer, and it does not read the state as a state at all. The spin state belongs to the rest frame it was set up in.

In [ ]:
# A boost along x and then along y: a lab moving through the lab at rest.
moving = (mv.xt * 0.4).exp() * (mv.yt * -0.3).exp()                             # [] Rotor
# The same experiment in the moving lab, and its steady state carried back to rest: the same state.
carried = steady(generator(0.7, 1.3, moving))                                    # [] State
at_rest = steady(rates)                                                          # [] State

# The state at rest, seen by the moving observer: in that observer's own frame, and split into the
# part it can read as a state, which its adjoint leaves unchanged, and the rest, I times a plane.
seen = moving << at_rest                                                         # [] State
readable = 0.5 * (seen + (mv.t >> seen.reverse()))                               # [] State
unreadable = seen - readable                                                     # [] State

In [ ]:
print("Bloch vector at rest:              ", render.components(at_rest))
print("in the moving lab, carried back:   ", render.components(moving << carried))
print("seen by the moving observer:       ", render.components(readable))
print("and the part it cannot read, over I:", render.components(unreadable / I))

In [ ]:
# checks
# The generator is the Bloch equations; the steady states do not change and are the textbook ones;
# the echo built as a map has lost only the true dephasing.
r = render.components(state(bloch))
bloch_equations = np.cross([1.3, 0.0, 0.7], r) - np.array([r[0] / T2, r[1] / T2, (r[2] - 1) / T1])
np.testing.assert_allclose(render.components(change), bloch_equations, atol=1e-12)
np.testing.assert_allclose(driven(settled).kernel, 0.0, atol=1e-12)
D, W = sweep[:, None], drives[None, :]
d = 1 + (D * T2) ** 2 + W**2 * T1 * T2
components = render.components(spectrum)
np.testing.assert_allclose(components[..., 2], (1 + (D * T2) ** 2) / d, atol=1e-12)
np.testing.assert_allclose(-components[..., 1], W * T2 / d, atol=1e-12)
np.testing.assert_allclose(render.transverse(echoed), np.exp(-times / T2), rtol=1e-5)
# Carrying the whole lab carries the steady state; a moving observer's reading has a part it cannot read.
np.testing.assert_allclose(carried.kernel, (moving >> at_rest).kernel, atol=1e-12)
assert np.abs(render.components(unreadable / I)).max() > 0.01